In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [87]:
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/task_2_data_ex - task_2_data_ex.csv.csv", delimiter=",")
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


**I assume each material has a consistent process every month. If the process changes, it must be assigned a different MaterialID.**
Therefore, I have grouped all stages of the process and calculated the total quantity for the entire year to eliminate the month field.

In [88]:
ds = data.copy()
ds['produced_material_quantity'] = pd.to_numeric(ds['produced_material_quantity'].str.replace(',', ''), errors='coerce')
ds['component_material_quantity'] = pd.to_numeric(ds['component_material_quantity'].str.replace(',', ''), errors='coerce')
ds

data_grouped = ds.groupby([
    'year',
    'produced_material',
    'produced_material_production_type',
    'produced_material_release_type',
    'component_material',
    'component_material_production_type',
    'component_material_release_type',
    'plant_id'
], dropna=False).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()

data_grouped.head(15)

,year,produced_material,produced_material_production_type,produced_material_release_type,component_material,component_material_production_type,component_material_release_type,plant_id,produced_material_quantity,component_material_quantity
0,2024,10000,8002,FIN,50000,8002.0,PROD,RLT_10,11708.0,11708.0
1,2024,10001,8002,FIN,50001,8002.0,PROD,RLT_10,12023.0,12023.0
2,2024,10002,8002,FIN,50002,8002.0,PROD,RLT_14,12067.0,12067.0
3,2024,10003,8002,FIN,50003,8002.0,PROD,RLT_14,12091.0,12091.0
4,2024,10004,8002,FIN,50004,8002.0,PROD,RLT_14,12091.0,12091.0
5,2024,10005,8002,FIN,50005,8002.0,PROD,RLT_14,12023.0,12023.0
6,2024,10006,8002,FIN,50006,8002.0,PROD,RLT_14,12067.0,12067.0
7,2024,10007,8002,FIN,50007,8002.0,PROD,RLT_16,12023.0,12023.0
8,2024,10008,8002,FIN,50008,8002.0,PROD,RLT_16,12067.0,12067.0
9,2024,10009,8002,FIN,50009,8002.0,PROD,RLT_14,12023.0,12023.0


Below, we can view all FIN materials and select the one to build the hierarchy for

In [89]:
fin_materials = data_grouped[data_grouped['produced_material_release_type'] == 'FIN'].copy()
print(fin_materials.loc[:,["plant_id","year", "produced_material", "produced_material_release_type"]])

  plant_id  year  produced_material produced_material_release_type
0   RLT_10  2024              10000                            FIN
1   RLT_10  2024              10001                            FIN
2   RLT_14  2024              10002                            FIN
3   RLT_14  2024              10003                            FIN
4   RLT_14  2024              10004                            FIN
5   RLT_14  2024              10005                            FIN
6   RLT_14  2024              10006                            FIN
7   RLT_16  2024              10007                            FIN
8   RLT_16  2024              10008                            FIN
9   RLT_14  2024              10009                            FIN


In the next block I define a Level 1 material for hierarchy.
Also to optimize performance, I subset the initial dataset by year and plant.


In [142]:
target_plant = 'RLT_10'
target_year = 2024
target_material = 10000

data_grouped_subset = data_grouped[(data_grouped["plant_id"]==target_plant) &
              (data_grouped["year"]==target_year) ].copy()

hierarchy= data_grouped_subset[(data_grouped_subset["produced_material"]==target_material) &
              (data_grouped_subset["year"]==target_year) &
              (data_grouped_subset["plant_id"]==target_plant) &
             (data_grouped_subset['produced_material_release_type'] == 'FIN')].copy()

hierarchy["level"] = 1
hierarchy['fin_material_id'] = hierarchy['produced_material']
hierarchy['fin_material_release_type'] = hierarchy['produced_material_release_type']
hierarchy['fin_material_production_type'] = hierarchy['produced_material_production_type']
hierarchy['fin_production_quantity'] = hierarchy['produced_material_quantity']


# data_grouped_subset.head()
hierarchy.head()


,year,produced_material,produced_material_production_type,produced_material_release_type,component_material,component_material_production_type,component_material_release_type,plant_id,produced_material_quantity,component_material_quantity,level,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity
0,2024,10000,8002,FIN,50000,8002.0,PROD,RLT_10,11708.0,11708.0,1,10000,FIN,8002,11708.0


Create a loop and UNION ALL to build a hierarchy

In [143]:
current_level = hierarchy.copy()

while True:
  next_level = data_grouped_subset.merge(
            current_level[['component_material', 'plant_id', 'year', 'level','fin_material_id','fin_material_release_type','fin_material_production_type','fin_production_quantity']],
            left_on=['produced_material', 'plant_id', 'year'],
            right_on=['component_material', 'plant_id', 'year'],
            suffixes=('', '_parent')
        ).drop(columns=['component_material_parent'])

  if next_level.empty:
     break

  next_level['level'] +=1

  hierarchy = pd.concat([hierarchy, next_level], ignore_index=True)
  current_level = next_level

hierarchy.head(20)


,year,produced_material,produced_material_production_type,produced_material_release_type,component_material,component_material_production_type,component_material_release_type,plant_id,produced_material_quantity,component_material_quantity,level,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity
0,2024,80000,8000,PROD,70000,NaN,RM,RLT_10,23478.0,30498.0,5,10000,FIN,8002,11708.0
1,2024,80000,8000,PROD,90005,NaN,ADD,RLT_10,23478.0,829.0,5,10000,FIN,8002,11708.0


rename columns

In [145]:
hierarchy = hierarchy.rename(columns={
    'plant_id':'plant',
    'produced_material':'prod_material_id',
    'produced_material_production_type':'prod_material_production_type',
    'produced_material_release_type':'prod_material_release_type',
    'produced_material_quantity':'prod_material_production_quantity',
    'component_material':'component_id',
    'component_material_production_type':'component_material_production_type',
    'component_material_release_type':'component_material_release_type',
    'component_material_quantity':'component_consumption_quantity'
})

#hierarchy.head(15)

,year,prod_material_id,prod_material_production_type,prod_material_release_type,component_id,component_material_production_type,component_material_release_type,plant,prod_material_production_quantity,component_consumption_quantity,level,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity
0,2024,10000,8002,FIN,50000,8002.0,PROD,RLT_10,11708.0,11708.0,1,10000,FIN,8002,11708.0
1,2024,50000,8002,PROD,80070,8007.0,PROD,RLT_10,9538.0,11303.0,2,10000,FIN,8002,11708.0
2,2024,50000,8002,PROD,90000,NaN,ADD,RLT_10,9538.0,598.0,2,10000,FIN,8002,11708.0
3,2024,50000,8002,PROD,90001,NaN,ADD,RLT_10,9538.0,242.0,2,10000,FIN,8002,11708.0
4,2024,80070,8007,PROD,80010,8001.0,PROD,RLT_10,11028.0,41769.0,3,10000,FIN,8002,11708.0
5,2024,80070,8007,PROD,90002,NaN,ADD,RLT_10,11028.0,355.0,3,10000,FIN,8002,11708.0
6,2024,80070,8007,PROD,90003,NaN,ADD,RLT_10,11028.0,121.0,3,10000,FIN,8002,11708.0
7,2024,80010,8001,PROD,80000,8000.0,PROD,RLT_10,21013.0,23360.0,4,10000,FIN,8002,11708.0
8,2024,80010,8001,PROD,90004,NaN,ADD,RLT_10,21013.0,1242.0,4,10000,FIN,8002,11708.0
9,2024,80000,8000,PROD,70000,NaN,RM,RLT_10,23478.0,30498.0,5,10000,FIN,8002,11708.0


hide the 1st row with fin_material, and arrange the columns in the requiered order

In [147]:
column_order = ['plant','fin_material_id','fin_material_release_type','fin_material_production_type','fin_production_quantity','prod_material_id','prod_material_release_type',
           'prod_material_production_type','prod_material_production_quantity', 'component_id', 'component_material_release_type', 'component_material_production_type', 'component_consumption_quantity', 'year', 'level']
hierarchy = hierarchy[column_order]


,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year,level
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024,2
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024,2
3,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024,2
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,2024,3
5,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,2024,3
6,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0,2024,3
7,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000.0,23360.0,2024,4
8,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0,2024,4
9,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,2024,5
10,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,90005,ADD,NaN,829.0,2024,5


In [150]:
result = hierarchy.iloc[1:]

result.head(20)

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year,level
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024,2
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024,2
3,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024,2
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,2024,3
5,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,2024,3
6,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0,2024,3
7,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000.0,23360.0,2024,4
8,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0,2024,4
9,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,2024,5
10,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,90005,ADD,NaN,829.0,2024,5
